In [11]:
import requests, json, os, datetime
import pandas as pd

In [12]:
URL = "https://raw.githubusercontent.com/alura-cursos/challenge2-data-science/refs/heads/main/TelecomX_Data.json"

In [13]:
resp = requests.get(URL, timeout=30)
resp.raise_for_status()
raw = resp.json()

In [14]:
ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
os.makedirs("data/raw", exist_ok=True)
with open(f"data/raw/telecomx_raw_{ts}.json", "w", encoding="utf-8") as f:
  json.dump(raw, f, ensure_ascii=False, indent=2)

In [15]:
df = pd.json_normalize(raw)
print(df.shape)
display(df.head())
display(df.info())
display(df.isna().sum())

In [16]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

In [17]:
date_cols = [c for c in df.columns if "date" in c or "signup" in c or "last" in c]
for c in date_cols:
  df[c] = pd.to_datetime(df[c], errors='coerce')

In [18]:
if 'signup_date' in df.columns:
  df['tenure_days'] = (pd.Timestamp.today() - df['signup_date']).dt.days
  df['tenure_months'] = (df['tenure_days'] / 30).round(1)

In [20]:
if 'churn' in df.columns:
  df['churn_flag'] = df['churn'].map({'Yes':1, 'No':0}).fillna(0).astype(int)

In [21]:
num_cols = df.select_dtypes(include='number').columns.tolist()
df[num_cols] = df[num_cols].fillna(df[num_cols].median())


In [22]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
for c in cat_cols:
    df[c] = df[c].str.strip().str.lower()

In [23]:
os.makedirs("data/processed", exist_ok=True)
df.to_parquet("data/processed/telecomx_transformed.parquet", index=False)
df.to_csv("data/processed/telecomx_transformed.csv", index=False)